# Meta Thinker: Data Loading and Preparation

This notebook demonstrates how to load and prepare datasets for training Meta Thinker models on various reasoning tasks.

## Overview

We'll cover:
1. Loading standard reasoning datasets (GSM8K, CommonsenseQA, LogiQA, Game24)
2. Understanding dataset structures
3. Preparing training data with different reasoning styles
4. Creating formatted datasets for model training

## Prerequisites

Install required packages:
```bash
pip install datasets transformers pyyaml
```

## 1. Dataset Loading

We'll load four key reasoning datasets used in Meta Thinker research.

### 1.1 GSM8K - Grade School Math Problems

GSM8K contains 8,500 grade-school math word problems requiring multi-step reasoning.

In [ ]:
from datasets import load_dataset
import json

# Load GSM8K dataset
gsm8k_dataset = load_dataset("gsm8k", "main")
gsm8k_train = gsm8k_dataset["train"]
gsm8k_test = gsm8k_dataset["test"]

print(f"GSM8K Training samples: {len(gsm8k_train)}")
print(f"GSM8K Test samples: {len(gsm8k_test)}")
print("\nExample question:")
print(gsm8k_train[0])

### 1.2 CommonsenseQA - Common Sense Reasoning

CommonsenseQA tests everyday knowledge and intuitive reasoning about the physical and social world.

In [ ]:
# Load CommonsenseQA dataset
commonsense_dataset = load_dataset("tau/commonsense_qa")
commonsense_train = commonsense_dataset['train']

print(f"CommonsenseQA Training samples: {len(commonsense_train)}")
print("\nExample question:")
example = commonsense_train[0]
print(f"Question: {example['question']}")
print(f"Choices: {example['choices']}")
print(f"Answer: {example['answerKey']}")

### 1.3 LogiQA - Logical Reasoning

LogiQA evaluates logical reasoning based on reading comprehension.

In [ ]:
# Load LogiQA dataset
logiqa_dataset = load_dataset("lucasmccabe/logiqa")
logiqa_train = logiqa_dataset['train']

print(f"LogiQA Training samples: {len(logiqa_train)}")
print("\nExample question:")
print(logiqa_train[0])

### 1.4 Game24 - Arithmetic Reasoning

The 24-Point Game requires using four numbers with arithmetic operations to reach 24.

In [ ]:
import pandas as pd

# Load Game24 dataset from local CSV
game24_df = pd.read_csv("./datasets/game24/game24_data.csv")

print(f"Game24 samples: {len(game24_df)}")
print("\nDataset preview:")
print(game24_df.head())

## 2. Loading Generated Responses

Load pre-generated responses from different reasoning styles.

In [ ]:
# Helper function to load JSON files
def load_json(file_path):
    """Load JSON data from file."""
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Load generated responses for different datasets
generated_data = {
    'commonsense': load_json("./input_data/data_commonsense.json"),
    'logiqa': load_json("./input_data/data_logiqa.json"),
    'gsm8k': load_json("./input_data/data_gsm8k.json")
}

# Display dataset sizes
for name, data in generated_data.items():
    print(f"{name}: {len(data)} samples")

## 3. Reasoning Styles Definition

Meta Thinker supports multiple reasoning paradigms:

- **CoT (Chain of Thought)**: Step-by-step linear reasoning
- **ToT (Tree of Thought)**: Multi-path parallel exploration
- **CoD (Chain of Draft)**: Concise minimal drafts
- **SoT (Sketch of Thought)**: Adaptive structured reasoning

In [ ]:
# Define reasoning style prompts
REASONING_STYLES = {
    'CoT': "Let's think step by step.",
    
    'ToT': (
        "Imagine three different experts are answering this question. "
        "All experts will write down 1 step of their thinking, then share it with the group. "
        "Then all experts will go on to the next step, etc. "
        "If any expert realises they're wrong at any point then they leave. "
        "The question is..."
    ),
    
    'CoD': (
        "Think step by step, but only keep minimum draft for each thinking step, "
        "with 5 words at most."
    ),
    
    'SoT': (
        "Sketch-of-Thought Reasoning Style:\n"
        "In this reasoning style, you are encouraged to sketch out your thinking using "
        "adaptive, concise, and structured traces, similar to how humans use mental sketches. "
        "Your reasoning should:\n"
        "- Focus on key steps and concepts rather than verbose justifications.\n"
        "- Use compact sketches to iteratively refine partial answers.\n"
        "- Employ error detection and correction, revising earlier steps if contradictions "
        "or mistakes are noticed.\n"
        "- Prioritize high information density, expressing reasoning in as few words as "
        "necessary while preserving clarity.\n\n"
        "Unlike full chain-of-thought explanations, Sketch-of-Thought favors efficiency "
        "and flexibility: use placeholders, abbreviations, or shorthand when appropriate, "
        "and allow yourself to adjust your sketch if better strategies or errors are "
        "discovered along the way.\n\n"
        "The final response should combine the refined sketch and a clear, conclusive answer."
    )
}

# Response format template
RESPONSE_FORMAT = """Answer the question in the following format:

<think>
[Explain your reasoning process. Reflect on failures and what you've learned.]
</think>

<answer>
[Only provide the final answer to the question]
</answer>
"""

print("Reasoning styles loaded successfully!")
print(f"Available styles: {list(REASONING_STYLES.keys())}")

## 4. Task Descriptions

Detailed descriptions of each reasoning task to provide context to the model.

In [ ]:
TASK_DESCRIPTIONS = {
    'gsm8k': (
        "You are a helpful assistant solving math problems from the GSM8K dataset. "
        "GSM8K is a collection of 8,500 high-quality, linguistically diverse grade-school "
        "math word problems. Designed to evaluate and enhance multi-step mathematical reasoning "
        "in language models, the dataset encompasses problems that require between two to eight "
        "steps to solve. Each problem primarily involves basic arithmetic operations—addition, "
        "subtraction, multiplication, and division—and is crafted to be solvable by a proficient "
        "middle school student. The solutions are provided in natural language, detailing each "
        "step to reach the final answer, thereby facilitating the development of models capable "
        "of human-like problem-solving processes."
    ),
    
    'commonsense': (
        "Commonsense Datasets are collections of questions and scenarios designed to evaluate "
        "a model's understanding of everyday knowledge and intuitive reasoning about the physical "
        "and social world. These datasets (e.g., CommonsenseQA, PIQA, SocialIQA) require Large "
        "Language Models (LLMs) to answer questions that humans typically resolve using common "
        "sense—implicit knowledge acquired through lived experience. Tasks often involve selecting "
        "the correct answer from multiple choices or generating coherent explanations for phenomena. "
        "Crucially, LLMs must not only retrieve factual knowledge but also infer unstated causal "
        "relationships, temporal sequences, and social norms, while avoiding implausible or "
        "inconsistent conclusions. Advanced evaluations further demand meta-reasoning—identifying "
        "when to apply strategies like Chain-of-Thought (CoT) for linear cause-effect problems or "
        "Tree-of-Thought (ToT) for multi-path scenarios—and even proposing novel reasoning "
        "frameworks for ambiguous or underspecified queries."
    ),
    
    'logiqa': (
        "You are answering questions from the LogiQA dataset, a benchmark designed to evaluate "
        "logical reasoning based on reading comprehension. Each question consists of a short "
        "passage, a question related to the passage, and four answer choices (A, B, C, D). "
        "Only one answer is logically correct, and your goal is to carefully reason over the "
        "passage and select the most logically valid option. The questions are often inspired "
        "by standardized logic exams and require deductive, abductive, or comparative reasoning. "
        "You should explain your reasoning clearly, step by step, before selecting your final answer."
    ),
    
    'game24': (
        "The 24-Point Game is a mathematical puzzle where the objective is to manipulate four "
        "given numbers using basic arithmetic operations—addition, subtraction, multiplication, "
        "and division—to achieve a result of 24. For instance, given the numbers 4, 9, 10, and 13, "
        "one possible solution is: (10 - 4) * (13 - 9) = 24. The dataset associated with this game "
        "includes numerous such combinations, each accompanied by solutions demonstrating the steps "
        "to reach 24. This dataset serves as a resource for developing and evaluating models' "
        "arithmetic reasoning and problem-solving abilities, particularly in formulating expressions "
        "that meet specific numerical targets."
    )
}

print("Task descriptions loaded successfully!")

## 5. Create Training Dataset

Format the data for model training by combining:
- Task descriptions
- Reasoning style instructions
- Questions and responses

The output format is compatible with standard LLM fine-tuning frameworks.

In [ ]:
def create_training_example(question, response, style, task_name):
    """
    Create a formatted training example.
    
    Args:
        question: The question/prompt
        response: The model's response
        style: The reasoning style used (CoT, ToT, etc.)
        task_name: Name of the task (gsm8k, commonsense, etc.)
    
    Returns:
        Dictionary with 'messages' key containing the formatted conversation
    """
    messages = [
        {
            'role': 'system',
            'content': (
                f"You are given a question from the {task_name} dataset. "
                f"{TASK_DESCRIPTIONS[task_name]} "
                f"This answer should be generated using the reasoning style '{style}'. "
                f"The style is described as: {REASONING_STYLES[style]}"
            )
        },
        {
            'role': 'user',
            'content': f"The question is: {question}"
        },
        {
            'role': 'assistant',
            'content': response
        }
    ]
    
    return {'messages': messages}

# Create training dataset
training_dataset = []
dataset_names = ['gsm8k', 'logiqa', 'commonsense']

print("Creating training dataset...\n")

for task_name in dataset_names:
    print(f"Processing {task_name}...")
    data = generated_data[task_name]
    
    for idx, item in enumerate(data):
        question = item['prompt']
        response = item['response']
        style = item['style']
        
        training_example = create_training_example(
            question=question,
            response=response,
            style=style,
            task_name=task_name
        )
        
        training_dataset.append(training_example)
    
    print(f"  Added {len(data)} examples from {task_name}")

print(f"\nTotal training examples: {len(training_dataset)}")

## 6. Save Training Dataset

Save the formatted training dataset to a JSON file for use in model training.

In [ ]:
# Save training dataset
output_path = "./input_data/training_dataset.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(training_dataset, f, ensure_ascii=False, indent=2)

print(f"✓ Training dataset saved to: {output_path}")
print(f"✓ Total examples: {len(training_dataset)}")

# Display example
print("\nExample training instance:")
print(json.dumps(training_dataset[0], indent=2)[:500] + "...")

## 7. Dataset Statistics

Analyze the distribution of reasoning styles and tasks in the training dataset.

In [ ]:
from collections import Counter

# Count reasoning styles
style_counts = Counter()
task_counts = Counter()

for task_name in dataset_names:
    data = generated_data[task_name]
    task_counts[task_name] = len(data)
    
    for item in data:
        style_counts[item['style']] += 1

print("Dataset Statistics:")
print("=" * 50)
print("\nExamples by Task:")
for task, count in task_counts.items():
    print(f"  {task:15s}: {count:5d} examples")

print("\nExamples by Reasoning Style:")
for style, count in style_counts.items():
    percentage = (count / len(training_dataset)) * 100
    print(f"  {style:15s}: {count:5d} examples ({percentage:.1f}%)")

print("\n" + "=" * 50)
print(f"Total: {len(training_dataset)} examples")

## Summary

This notebook demonstrated:

1. ✓ Loading multiple reasoning datasets (GSM8K, CommonsenseQA, LogiQA, Game24)
2. ✓ Understanding dataset structures and content
3. ✓ Defining reasoning styles (CoT, ToT, CoD, SoT)
4. ✓ Creating formatted training data with task descriptions
5. ✓ Saving the training dataset for model fine-tuning

## Next Steps

- **Notebook 01**: Meta-learning demonstrations
- **Notebook 02**: Game24 reasoning experiments
- **Notebook 03**: Testing and evaluation

The generated `training_dataset.json` can be used directly with standard LLM training frameworks like Hugging Face Transformers or the HALOs repository.